the file for the data explorationa nd usage 


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_classif
from scipy.stats import mannwhitneyu
import tensorflow as tf 

# optional
from sklearn.preprocessing import LabelEncoder

In [ ]:
df = pd.read_csv(r'C:\Users\shiva\OneDrive\Desktop\projects 2.0\ANN smart grid\smart-grid-fault-ANN\dataset\smart_grid_stability_augmented.csv')

In [ ]:
df.shape, df.dtypes, df.isnull().sum()

In [ ]:
#loading and inspection 

print("Initial Shape:", df.shape)

# Drop 'stab' if exists
if 'stab' in df.columns:
    df = df.drop(columns=['stab'])
    print("Dropped 'stab' column")

# Assume last column is target (change if needed)
TARGET = df.columns[-1]
FEATURES = df.columns.drop(TARGET)

print("Target:", TARGET)
print("Features:", list(FEATURES))

# Encode categorical if needed
if df[TARGET].dtype == 'object':
    le = LabelEncoder()
    df[TARGET] = le.fit_transform(df[TARGET])


Initial Shape: (60000, 14)
Dropped 'stab' column
Target: stabf
Features: ['tau1', 'tau2', 'tau3', 'tau4', 'p1', 'p2', 'p3', 'p4', 'g1', 'g2', 'g3', 'g4']


In [12]:
# Drop 'stab' if exists
if 'stab' in df.columns:
    df = df.drop(columns=['stab'])
    print("Dropped 'stab' column")

TARGET = df.columns[-1]
FEATURES = df.columns.drop(TARGET)

print("Target:", TARGET)
print("Features:", list(FEATURES))

df.info()

Target: stabf
Features: ['tau1', 'tau2', 'tau3', 'tau4', 'p1', 'p2', 'p3', 'p4', 'g1', 'g2', 'g3', 'g4']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   tau1    60000 non-null  float64
 1   tau2    60000 non-null  float64
 2   tau3    60000 non-null  float64
 3   tau4    60000 non-null  float64
 4   p1      60000 non-null  float64
 5   p2      60000 non-null  float64
 6   p3      60000 non-null  float64
 7   p4      60000 non-null  float64
 8   g1      60000 non-null  float64
 9   g2      60000 non-null  float64
 10  g3      60000 non-null  float64
 11  g4      60000 non-null  float64
 12  stabf   60000 non-null  int64  
dtypes: float64(12), int64(1)
memory usage: 6.0 MB


In [ ]:
class_counts = df[TARGET].value_counts()
print(class_counts)

plt.figure()
plt.pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%')
plt.title("Class Balance")
plt.show()

# TensorFlow-ready class weights
total = len(df)
num_classes = len(class_counts)

class_weight = {
    int(cls): total / (num_classes * count)
    for cls, count in class_counts.items()
}

print("Class Weights:", class_weight)

In [ ]:
groups = {
    "tau": [col for col in FEATURES if col.startswith("tau")],
    "p": [col for col in FEATURES if col.startswith("p")],
    "g": [col for col in FEATURES if col.startswith("g")]
}

groups

In [ ]:
for group_name, cols in groups.items():
    for col in cols:
        plt.figure()
        
        for cls in class_counts.index:
            subset = df[df[TARGET] == cls]
            plt.hist(subset[col], bins=30, alpha=0.5, label=f"class {cls}")
        
        plt.title(f"{group_name} - {col}")
        plt.legend()
        plt.show()

In [ ]:
mw_results = []
classes = df[TARGET].unique()

if len(classes) == 2:
    class_0 = df[df[TARGET] == classes[0]]
    class_1 = df[df[TARGET] == classes[1]]

    for col in FEATURES:
        stat, p = mannwhitneyu(class_0[col], class_1[col])
        mw_results.append((col, p))

    mw_results.sort(key=lambda x: x[1])

    print("Top Separating Features:")
    for col, p in mw_results[:10]:
        print(f"{col}: p-value = {p:.6f}")

In [ ]:
corr = df.corr()

plt.figure(figsize=(10,8))
plt.imshow(corr, interpolation='none')
plt.colorbar()

plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)

plt.title("Correlation Matrix")
plt.show()

target_corr = corr[TARGET].drop(TARGET).sort_values(ascending=False)

print("Top Correlations with Target:")
print(target_corr.head(10))

In [ ]:
cols = list(FEATURES)

fig, axes = plt.subplots(3, 4, figsize=(15,10))

for i, col in enumerate(cols[:12]):
    r = i // 4
    c = i % 4
    axes[r, c].boxplot(df[col].dropna())
    axes[r, c].set_title(col)

plt.tight_layout()
plt.show()

In [ ]:
X = df[FEATURES].values
y = df[TARGET].values

mi = mutual_info_classif(X, y)

mi_scores = pd.Series(mi, index=FEATURES).sort_values(ascending=False)

print(mi_scores)

plt.figure()
mi_scores.plot(kind='bar')
plt.title("Mutual Information")
plt.show()

In [ ]:
top_features = mi_scores.head(4).index.tolist()

sample_df = df[top_features + [TARGET]].sample(n=min(2000, len(df)))

for i in range(len(top_features)):
    for j in range(len(top_features)):
        plt.figure()
        plt.scatter(sample_df[top_features[i]],
                    sample_df[top_features[j]],
                    c=sample_df[TARGET],
                    cmap='viridis')
        
        plt.xlabel(top_features[i])
        plt.ylabel(top_features[j])
        plt.title(f"{top_features[i]} vs {top_features[j]}")
        plt.show()

In [ ]:
X_tf = tf.convert_to_tensor(df[FEATURES].values, dtype=tf.float32)
y_tf = tf.convert_to_tensor(df[TARGET].values, dtype=tf.float32)

print("X shape:", X_tf.shape)
print("y shape:", y_tf.shape)

In [ ]:
print("===== EDA SUMMARY =====")

print(f"Dataset Shape: {df.shape}")
print(f"Target: {TARGET}")

print("\nClass Balance:")
print(class_counts)

print("\nTop Features (Mutual Info):")
print(mi_scores.head(5))

if len(classes) == 2:
    print("\nBest Separating Features:")
    print(mw_results[:5])

print("\nTop Correlations:")
print(target_corr.head(5))

print("\nNext Steps:")
print("- Normalize data (VERY IMPORTANT)")
print("- Train TensorFlow ANN")
print("- Use class_weight")